# Climatological Stations Analysis - EH 26289

This notebook processes daily precipitation data from 16 climatological stations in the Río Tempoal and surrounding basins. The analysis includes:
1. Downloading daily precipitation data from CONAGUA URLs
2. Filtering stations with data extending to 2015 and at least 20 years of records
3. Creating monthly precipitation DataFrames with year index and month columns

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
import urllib3

# Disable SSL verification warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## 2. Load and Explore Estaciones CSV

In [3]:
# Read the estaciones.csv file
estaciones = pd.read_csv('../data/precipitacion/estaciones.csv')

# Check the DIARIOS column
print("DIARIOS column sample:")
print(estaciones[['CLAVE', 'NOMBRE', 'DIARIOS']].head(10))
print(f"\nTotal rows: {len(estaciones)}")
print(f"Non-null DIARIOS entries: {estaciones['DIARIOS'].notna().sum()}")

DIARIOS column sample:
   CLAVE                       NOMBRE  \
0  13011                     HUEJUTLA   
1  13015   SAN AGUSTIN METZIQUITITLAN   
2  13042            ZACUALTIPAN (SMN)   
3  13077                   METZTITLAN   
4  13087                SAN CRISTOBAL   
5  13093                      VENADOS   
6  13135                    ATLAPEXCO   
7  13139                HUAUTLA (DGE)   
8  30016                BENITO JUAREZ   
9  30041  CHICONTEPEC DE TEJEDA (SMN)   

                                             DIARIOS  
0  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
1  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
2  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
3  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
4  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
5  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
6  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
7  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
8  https://smn.conagua.gob.mx/

## 3. Download Daily Precipitation Files from DIARIOS URLs

In [21]:
'''# Read the estaciones.csv file
estaciones = pd.read_csv('data/precipitacion/estaciones.csv')

# Target folder
target_folder = Path('data/precipitacion')
target_folder.mkdir(parents=True, exist_ok=True)

# Download each file
download_count = 0
error_count = 0

for idx, row in estaciones.iterrows():
    url = row['DIARIOS']
    clave = row['CLAVE']
    
    if pd.isna(url):
        print(f"Skipping row {idx}: No URL for {clave}")
        continue
    
    try:
        # Extract filename from URL
        filename = urlparse(url).path.split('/')[-1]
        filepath = target_folder / filename
        
        print(f"Downloading {clave}: {filename}...", end=' ')
        
        # Download the file with SSL verification disabled
        response = requests.get(url, timeout=10, verify=False)
        response.raise_for_status()
        
        # Save the file (overwriting if it exists)
        with open(filepath, 'wb') as f:
            f.write(response.content)
        
        print(f"✓ ({len(response.content)} bytes)")
        download_count += 1
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        error_count += 1

print(f"\n--- Summary ---")
print(f"Successfully downloaded: {download_count}")
print(f"Errors: {error_count}")
print(f"Total: {len(estaciones)}")'''

'# Read the estaciones.csv file\nestaciones = pd.read_csv(\'data/precipitacion/estaciones.csv\')\n\n# Target folder\ntarget_folder = Path(\'data/precipitacion\')\ntarget_folder.mkdir(parents=True, exist_ok=True)\n\n# Download each file\ndownload_count = 0\nerror_count = 0\n\nfor idx, row in estaciones.iterrows():\n    url = row[\'DIARIOS\']\n    clave = row[\'CLAVE\']\n\n    if pd.isna(url):\n        print(f"Skipping row {idx}: No URL for {clave}")\n        continue\n\n    try:\n        # Extract filename from URL\n        filename = urlparse(url).path.split(\'/\')[-1]\n        filepath = target_folder / filename\n\n        print(f"Downloading {clave}: {filename}...", end=\' \')\n\n        # Download the file with SSL verification disabled\n        response = requests.get(url, timeout=10, verify=False)\n        response.raise_for_status()\n\n        # Save the file (overwriting if it exists)\n        with open(filepath, \'wb\') as f:\n            f.write(response.content)\n\n        pr

## 4. Analyze Climatological Stations with Data Extending to 2015

In [4]:
# Read the estaciones.csv
estaciones = pd.read_csv('../data/precipitacion/estaciones.csv')

# Filter for climatological stations only
climatologicas = estaciones[estaciones['TIPO_EST'] == 'CLIMATOLÓGICA'].copy()

print(f"Total climatological stations: {len(climatologicas)}")
print(f"\nAnalyzing daily data files for year range...\n")

# Check each file
results = []
data_folder = Path('../data/precipitacion')

for idx, row in climatologicas.iterrows():
    clave = row['CLAVE']
    nombre = row['NOMBRE']
    filename = f"dia{clave}.txt"
    filepath = data_folder / filename
    
    if not filepath.exists():
        print(f"❌ {clave}: File not found")
        continue
    
    try:
        # Read the file
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        # Find lines that look like dates
        dates = []
        for line in lines:
            parts = line.split()
            if len(parts) > 0:
                date_str = parts[0]
                try:
                    # Try parsing as YYYY/MM/DD
                    if '/' in date_str:
                        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
                        dates.append(date_obj)
                    elif '-' in date_str and len(date_str) == 10:
                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                        dates.append(date_obj)
                except:
                    pass
        
        if dates:
            min_year = min(dates).year
            max_year = max(dates).year
            
            has_2015 = any(d.year >= 2015 for d in dates)
            
            status = "✓" if has_2015 else "✗"
            print(f"{status} {clave}: {nombre:40s} | {min_year} - {max_year}")
            
            if has_2015:
                results.append({
                    'CLAVE': clave,
                    'NOMBRE': nombre,
                    'MIN_YEAR': min_year,
                    'MAX_YEAR': max_year
                })
        else:
            print(f"❓ {clave}: {nombre:40s} | Could not parse dates")
    
    except Exception as e:
        print(f"❌ {clave}: Error - {str(e)[:50]}")

print(f"\n{'='*70}")
print(f"Climatological stations with data including 2015 or later:")
print(f"{'='*70}")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nTotal selected: {len(results_df)}")

Total climatological stations: 28

Analyzing daily data files for year range...

✓ 13011: HUEJUTLA                                 | 1969 - 2026
✓ 13015: SAN AGUSTIN METZIQUITITLAN               | 1942 - 2025
✓ 13042: ZACUALTIPAN (SMN)                        | 1941 - 2026
✓ 13077: METZTITLAN                               | 1952 - 2026
✓ 13087: SAN CRISTOBAL                            | 1954 - 2025
✓ 13093: VENADOS                                  | 1951 - 2026
✓ 13135: ATLAPEXCO                                | 1981 - 2026
✓ 13139: HUAUTLA (DGE)                            | 1981 - 2025
✓ 30016: BENITO JUAREZ                            | 1962 - 2022
✓ 30041: CHICONTEPEC DE TEJEDA (SMN)              | 1947 - 2023
✓ 30067: HUAYACOCOTLA                             | 1961 - 2026
✓ 30098: LOS HULES                                | 1960 - 2026
✓ 30180: TERRERILLOS                              | 1961 - 2026
✓ 30382: ZONTECOMATLAN                            | 1983 - 2019
✗ 13003: CALNALI       

## 5. Select Stations with at Least 20 Years of Records

In [40]:
# Read the estaciones.csv
estaciones = pd.read_csv('../data/precipitacion/estaciones.csv')

# Filter for climatological stations only
climatologicas = estaciones[estaciones['TIPO_EST'] == 'CLIMATOLÓGICA'].copy()

print(f"Total climatological stations: {len(climatologicas)}")
print(f"\nAnalyzing unique years in each station's data...\n")

# Check each file
results = []
data_folder = Path('../data/precipitacion')

for idx, row in climatologicas.iterrows():
    clave = row['CLAVE']
    nombre = row['NOMBRE']
    filename = f"dia{clave}.txt"
    filepath = data_folder / filename

    if not filepath.exists():
        print(f"❌ {clave}: File not found")
        continue

    try:
        # Read the file
        with open(filepath, 'r') as f:
            lines = f.readlines()

        # Extract unique years from all data lines
        unique_years = set()
        for line in lines:
            parts = line.split()
            if len(parts) > 0:
                date_str = parts[0]
                try:
                    # Try parsing as YYYY/MM/DD
                    if '/' in date_str:
                        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
                        unique_years.add(date_obj.year)
                    elif '-' in date_str and len(date_str) == 10:
                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                        unique_years.add(date_obj.year)
                except:
                    pass

        if unique_years:
            num_years = len(unique_years)
            min_year = min(unique_years)
            max_year = max(unique_years)
            span = max_year - min_year + 1

            has_20_years = num_years >= 20

            status = "✓" if has_20_years else "✗"
            print(f"{status} {clave}: {nombre:40s} | {num_years:2d} unique years ({min_year}-{max_year}, span: {span} years)")

            if has_20_years:
                results.append({
                    'CLAVE': clave,
                    'NOMBRE': nombre,
                    'UNIQUE_YEARS': num_years,
                    'MIN_YEAR': min_year,
                    'MAX_YEAR': max_year,
                    'SPAN': span
                })
        else:
            print(f"❓ {clave}: {nombre:40s} | Could not parse dates")

    except Exception as e:
        print(f"❌ {clave}: Error - {str(e)[:50]}")

print(f"\n{'='*80}")
print("Climatological stations with at least 20 years of records (continuous or discontinuous):")
print(f"{'='*80}")
results_df = pd.DataFrame(results)
if len(results_df) > 0:
    results_df = results_df.sort_values('UNIQUE_YEARS', ascending=False)
    print(results_df.to_string(index=False))
    print(f"\nTotal selected: {len(results_df)}")
else:
    print("No stations with 20+ years of records found.")

Total climatological stations: 28

Analyzing unique years in each station's data...

✓ 13011: HUEJUTLA                                 | 58 unique years (1969-2026, span: 58 years)
✓ 13015: SAN AGUSTIN METZIQUITITLAN               | 53 unique years (1942-2025, span: 84 years)
✓ 13042: ZACUALTIPAN (SMN)                        | 86 unique years (1941-2026, span: 86 years)
✓ 13077: METZTITLAN                               | 69 unique years (1952-2026, span: 75 years)
✓ 13087: SAN CRISTOBAL                            | 72 unique years (1954-2025, span: 72 years)
✓ 13093: VENADOS                                  | 72 unique years (1951-2026, span: 76 years)
✓ 13135: ATLAPEXCO                                | 44 unique years (1981-2026, span: 46 years)
✓ 13139: HUAUTLA (DGE)                            | 45 unique years (1981-2025, span: 45 years)
✓ 30016: BENITO JUAREZ                            | 53 unique years (1962-2022, span: 61 years)
✓ 30041: CHICONTEPEC DE TEJEDA (SMN)              |

## 6. Final Selection: Stations with 2015+ Data AND 20+ Years of Records

In [ ]:
# The 16 selected stations meeting both criteria:
# - Data extending to at least 2015
# - At least 20 years of records (continuous or discontinuous)

selected_stations = [
    '13042', '30041', '13087', '13093', '13077', '30180', 
    '30098', '13011', '13015', '30016', '30067', '13139', 
    '13135', '30359', '13137', '30382'
]

print(f"Selected {len(selected_stations)} climatological stations:")
print(selected_stations)

## 7. Create Monthly Precipitation DataFrames

In [2]:
# List of the 16 selected stations
selected_stations = [
    '13042', '30041', '13087', '13093', '13077', '30180', 
    '30098', '13011', '13015', '30016', '30067', '13139', 
    '13135', '30359', '13137', '30382'
]

data_folder = Path('../data/precipitacion')
dfs_monthly = {}

print(f"Processing {len(selected_stations)} stations...\n")

for clave in selected_stations:
    filename = f"dia{clave}.txt"
    filepath = data_folder / filename
    
    if not filepath.exists():
        print(f"❌ {clave}: File not found")
        continue
    
    try:
        # Read the file
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        # Parse the data - extract dates and precipitation values
        dates = []
        precip = []
        
        for line in lines:
            parts = line.split()
            if len(parts) >= 2:
                date_str = parts[0]
                try:
                    # Try parsing as YYYY/MM/DD
                    if '/' in date_str:
                        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
                    elif '-' in date_str and len(date_str) == 10:
                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                    else:
                        continue
                    
                    # Try to get precipitation value (usually second column)
                    try:
                        precip_val = float(parts[1])
                    except:
                        # If precipitation is in another column, try other positions
                        precip_val = None
                        for i in range(1, min(len(parts), 5)):
                            try:
                                val = float(parts[i])
                                precip_val = val
                                break
                            except:
                                continue
                    
                    if precip_val is not None:
                        dates.append(date_obj)
                        precip.append(precip_val)
                except:
                    pass
        
        if dates and precip:
            # Create daily dataframe
            df_daily = pd.DataFrame({
                'date': dates,
                'precipitation': precip
            })
            df_daily['date'] = pd.to_datetime(df_daily['date'])
            df_daily = df_daily.set_index('date')
            
            # Resample to monthly sum
            df_monthly = df_daily.resample('ME').sum()
            
            # Add year and month columns
            df_monthly['year'] = df_monthly.index.year
            df_monthly['month'] = df_monthly.index.month
            
            # Pivot to get year as index and month as columns
            pivot_df = df_monthly.pivot(index='year', columns='month', values='precipitation')
            
            # Store it
            dfs_monthly[clave] = pivot_df
            
            print(f"✓ {clave}: {pivot_df.shape[0]} years × 12 months")
        else:
            print(f"❌ {clave}: Could not parse precipitation data")
    
    except Exception as e:
        print(f"❌ {clave}: Error - {str(e)[:50]}")

print(f"\n{'='*60}")
print(f"Successfully created {len(dfs_monthly)} monthly DataFrames")
print(f"{'='*60}")

Processing 16 stations...

✓ 13042: 86 years × 12 months
✓ 30041: 77 years × 12 months
✓ 13087: 72 years × 12 months
✓ 13093: 76 years × 12 months
✓ 13077: 75 years × 12 months
✓ 30180: 66 years × 12 months
✓ 30098: 67 years × 12 months
✓ 13011: 58 years × 12 months
✓ 13015: 84 years × 12 months
✓ 30016: 61 years × 12 months
✓ 30067: 66 years × 12 months
✓ 13139: 45 years × 12 months
✓ 13135: 46 years × 12 months
✓ 30359: 44 years × 12 months
✓ 13137: 35 years × 12 months
✓ 30382: 37 years × 12 months

Successfully created 16 monthly DataFrames


## 8. Sample Monthly Precipitation DataFrames

In [3]:
# Show sample of each monthly dataframe
for clave in list(dfs_monthly.keys())[:3]:
    print(f"\nStation {clave}:")
    print(f"Shape: {dfs_monthly[clave].shape}")
    print("\nFirst 5 years:")
    print(dfs_monthly[clave].head())
    print("\n" + "="*80)


Station 13042:
Shape: (86, 12)

First 5 years:
month     1       2      3      4      5       6       7       8       9   \
year                                                                        
1941     NaN     NaN    NaN    NaN    NaN     NaN     NaN     NaN     NaN   
1942   43.17   25.50  84.03   0.00   0.00    0.00  311.02  185.00  599.02   
1943   64.00   24.01  28.02  39.03  36.01  141.04   43.02  160.03  266.01   
1944    0.00  172.04  40.10   0.05  93.06  278.05   81.02  318.02  449.02   
1945   15.18   21.03  33.04   5.52  33.03   88.00   65.84  159.05   79.66   

month      10      11     12  
year                          
1941    45.06  186.65  80.00  
1942    39.02  133.01  35.00  
1943   126.09   43.03  36.02  
1944   147.02   46.57  41.08  
1945    78.92   56.32  18.00  


Station 30041:
Shape: (77, 12)

First 5 years:
month     1     2      3      4      5      6      7      8      9      10  \
year                                                                

## Summary

This notebook successfully:
1. **Downloaded 29 daily precipitation files** from CONAGUA DIARIOS URLs
2. **Identified 16 climatological stations** with data extending to at least 2015 and at least 20 years of records
3. **Created monthly precipitation DataFrames** for each station with:
   - Year as index
   - Months (1-12) as columns
   - Monthly precipitation totals (mm) as values

The selected stations span from 1941 to 2026, providing a comprehensive dataset for hydrological analysis in the EH 26289 basin.

In [4]:
for clave in list(dfs_monthly.keys()):
    df = dfs_monthly[clave].copy()

    # Remove previously added summary row/column if present
    if "media_mensual" in df.index:
        df = df.drop(index="media_mensual")
    if "anual" in df.columns:
        df = df.drop(columns=["anual"])

    # Keep only complete years
    df = df.dropna()

    # Drop years with at least one rainy-season month equal to zero (May-Sep)
    rainy_cols = [m for m in [5, 6, 7, 8, 9] if m in df.columns]
    if rainy_cols:
        df = df.loc[~df[rainy_cols].eq(0).any(axis=1)].copy()

    # Recompute annual and mean row
    df["anual"] = df.sum(axis=1)
    df.loc["media_mensual"] = df.mean(numeric_only=True)

    dfs_monthly[clave] = df

In [7]:
excedencias = {}
for clave in list(dfs_monthly.keys()):
    excedencia = pd.DataFrame(dfs_monthly[clave]["anual"].sort_values(ascending=False))
    excedencia.index = np.arange(1, len(excedencia) + 1)
    excedencia.columns = ["Precip(mm)"]
    excedencia["Tr"] = (len(excedencia) + 1) / excedencia.index
    excedencia["Pe"] = 1 / excedencia["Tr"]
    excedencias[clave] = excedencia

In [23]:
# Find Precip(mm) corresponding to Pe = 0.85 for each station (linear interpolation where needed)
target_pe = 0.85
precip_at_pe85 = {}
details = {}
for clave, df in excedencias.items():
    # Work on a copy and drop NA rows
    edf = df[['Precip(mm)', 'Pe']].dropna().copy()
    edf = edf.sort_values('Pe').reset_index(drop=True)
    pe_vals = edf['Pe'].values
    p_vals = edf['Precip(mm)'].values
    n = len(pe_vals)
    if n == 0:
        precip_at_pe85[clave] = np.nan
        details[clave] = {'status': 'no_data'}
        continue
    # If exact match exists, take it
    mask_eq = np.isclose(pe_vals, target_pe)
    if mask_eq.any():
        precip_at_pe85[clave] = float(p_vals[mask_eq][0])
        details[clave] = {'status': 'exact'}
        continue
    # If target_pe is outside the empirical range, set NaN (no reliable interpolation)
    if target_pe < pe_vals.min() or target_pe > pe_vals.max():
        precip_at_pe85[clave] = np.nan
        details[clave] = {'status': 'out_of_range', 'min_pe': float(pe_vals.min()), 'max_pe': float(pe_vals.max())}
        continue
    # Locate insertion point and linearly interpolate between neighbouring points
    idx = np.searchsorted(pe_vals, target_pe)
    if idx == 0:
        precip_at_pe85[clave] = float(p_vals[0])
        details[clave] = {'status': 'clamped_low'}
    elif idx >= n:
        precip_at_pe85[clave] = float(p_vals[-1])
        details[clave] = {'status': 'clamped_high'}
    else:
        pe_lo, pe_hi = pe_vals[idx-1], pe_vals[idx]
        p_lo, p_hi = p_vals[idx-1], p_vals[idx]
        # Linear interpolation in (Pe, Precip) space
        t = (target_pe - pe_lo) / (pe_hi - pe_lo)
        precip_interp = p_lo + t * (p_hi - p_lo)
        precip_at_pe85[clave] = float(precip_interp)
        details[clave] = {'status': 'interpolated', 'pe_lo': float(pe_lo), 'pe_hi': float(pe_hi), 'p_lo': float(p_lo), 'p_hi': float(p_hi)}

# Create a summary DataFrame
summary = pd.DataFrame([{'CLAVE': k, 'Precip_at_Pe85_mm': v, 'detail': details.get(k)} for k, v in precip_at_pe85.items()])
summary = summary.set_index('CLAVE').sort_index()
print('Precip(mm) at Pe=0.85 for each station:')
print(summary[['Precip_at_Pe85_mm']])

# Keep results available in the notebook namespace
precip_at_pe85_dict = precip_at_pe85
excedencias_pe85_summary = summary

# Optionally, save to CSV in data folder
try:
    summary.to_csv('../data/precipitacion/excedencias_pe85_summary.csv')
    print('Saved summary to ../data/precipitacion/excedencias_pe85_summary.csv')
except Exception as e:
    print('Could not save summary:', e)

Precip(mm) at Pe=0.85 for each station:
       Precip_at_Pe85_mm
CLAVE                   
13011          1006.2050
13015           252.2940
13042           946.9500
13077           298.0700
13087           254.7540
13093           275.1105
13135          1097.2795
13137           945.1000
13139          1083.5800
30016           240.8600
30041          1403.3000
30067          1037.5300
30098           986.7460
30180          1105.7580
30359           548.6300
30382          1776.1620
Saved summary to ../data/precipitacion/excedencias_pe85_summary.csv


In [ ]:
f = "../data/precipitacion/areasThiessen_corregido.csv"
area = pd.read_csv(f)
area = area.set_index("CLAVE")

,name,folders,descriptio,altitude,alt_mode,time_begin,time_end,time_when,ALTITUD,CUENCA,...,NOMBRE,NORMALES_1,NORMALES_2,NORMALES_3,NORMALES_4,ORG_CUENCA,SITUACION,TIPO_EST,AREA_KM2,A_km2
CLAVE,,,,,,,,,,,,,,,,,,,,,
13042,13042,Red Nacional de Estaciones Climatológicas Conv...,NaN,1969.0,NaN,NaN,NaN,NaN,1969,RÍO METZTITLÁN 2,...,ZACUALTIPAN (SMN),https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,224,223.992565
13015,13015,Red Nacional de Estaciones Climatológicas Conv...,NaN,1373.0,NaN,NaN,NaN,NaN,1373,RÍO METZTITLÁN 2,...,SAN AGUSTIN METZIQUITITLAN,NaN,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,12,11.685194
30359,30359,Red Nacional de Estaciones Climatológicas Conv...,NaN,2266.0,NaN,NaN,NaN,NaN,2266,RÍO TUXPAN,...,PALO BENDITO,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO CENTRO,SUSPENDIDA,CLIMATOLÓGICA,15,15.483681
30067,30067,Red Nacional de Estaciones Climatológicas Conv...,NaN,2168.0,NaN,NaN,NaN,NaN,2168,RÍO TUXPAN,...,HUAYACOCOTLA,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,NaN,NaN,NaN,GOLFO CENTRO,OPERANDO,CLIMATOLÓGICA,112,112.475988
13135,13135,Red Nacional de Estaciones Climatológicas Conv...,NaN,168.0,NaN,NaN,NaN,NaN,168,RÍO LOS HULES,...,ATLAPEXCO,NaN,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,39,38.807255


In [12]:
area["w"] = area["A_km2"] / area["A_km2"].sum()
area

,name,folders,descriptio,altitude,alt_mode,time_begin,time_end,time_when,ALTITUD,CUENCA,...,NORMALES_1,NORMALES_2,NORMALES_3,NORMALES_4,ORG_CUENCA,SITUACION,TIPO_EST,AREA_KM2,A_km2,w
CLAVE,,,,,,,,,,,,,,,,,,,,,
13042,13042,Red Nacional de Estaciones Climatológicas Conv...,NaN,1969.0,NaN,NaN,NaN,NaN,1969,RÍO METZTITLÁN 2,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,224,223.992565,0.148582
13015,13015,Red Nacional de Estaciones Climatológicas Conv...,NaN,1373.0,NaN,NaN,NaN,NaN,1373,RÍO METZTITLÁN 2,...,NaN,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,12,11.685194,0.007751
30359,30359,Red Nacional de Estaciones Climatológicas Conv...,NaN,2266.0,NaN,NaN,NaN,NaN,2266,RÍO TUXPAN,...,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO CENTRO,SUSPENDIDA,CLIMATOLÓGICA,15,15.483681,0.010271
30067,30067,Red Nacional de Estaciones Climatológicas Conv...,NaN,2168.0,NaN,NaN,NaN,NaN,2168,RÍO TUXPAN,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,NaN,NaN,NaN,GOLFO CENTRO,OPERANDO,CLIMATOLÓGICA,112,112.475988,0.074609
13135,13135,Red Nacional de Estaciones Climatológicas Conv...,NaN,168.0,NaN,NaN,NaN,NaN,168,RÍO LOS HULES,...,NaN,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,39,38.807255,0.025742
30382,30382,Red Nacional de Estaciones Climatológicas Conv...,NaN,1612.0,NaN,NaN,NaN,NaN,1612,RÍO CALABOZO,...,NaN,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,581,581.171406,0.385510
30016,30016,Red Nacional de Estaciones Climatológicas Conv...,NaN,2202.0,NaN,NaN,NaN,NaN,2202,RÍO CALABOZO,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,311,311.383948,0.206551
13139,13139,Red Nacional de Estaciones Climatológicas Conv...,NaN,503.0,NaN,NaN,NaN,NaN,503,RÍO LOS HULES,...,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,98,98.247370,0.065171
30041,30041,Red Nacional de Estaciones Climatológicas Conv...,NaN,291.0,NaN,NaN,NaN,NaN,291,RÍO CALABOZO,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,81,81.163964,0.053839


In [22]:
# Build resumen using only CLAVEs present in `area`
# Normalize index types
summary.index = summary.index.astype(str)
area.index = area.index.astype(str)

# Find intersection of CLAVEs present in area and summary
common_claves = area.index.intersection(summary.index)
resumen = summary.loc[common_claves, ['Precip_at_Pe85_mm']].copy()

# Identify numeric area column
area_col = None
candidates = ['A_km2', 'AREA_KM2']
for col in candidates:
    if col in area.columns:
        area_col = col
        break
if area_col is None:
    numeric_cols = area.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        area_col = numeric_cols[-1]
if area_col is None:
    raise RuntimeError('Could not find numeric area column in `area`')

# Subset area to common claves and compute w normalized over that subset
area_sub = area.loc[common_claves].copy()
area_sub['w'] = area_sub[area_col] / area_sub[area_col].sum()

# Join weights
resumen = resumen.join(area_sub[['w']], how='left')
resumen.index.name = 'CLAVE'

# Save
out_dir = Path('../data/precipitacion')
out_dir.mkdir(parents=True, exist_ok=True)
resumen.to_csv(out_dir / 'resumen_pe85_w.csv')
print('Saved resumen to', out_dir / 'resumen_pe85_w.csv')
print(resumen)
resumen

Saved resumen to ../data/precipitacion/resumen_pe85_w.csv
       Precip_at_Pe85_mm         w
CLAVE                             
13042           946.9500  0.148582
13015           252.2940  0.007751
30359           548.6300  0.010271
30067          1037.5300  0.074609
13135          1097.2795  0.025742
30382          1776.1620  0.385510
30016           240.8600  0.206551
13139          1083.5800  0.065171
30041          1403.3000  0.053839
30180          1105.7580  0.021974


,Precip_at_Pe85_mm,w
CLAVE,,
13042,946.9500,0.148582
13015,252.2940,0.007751
30359,548.6300,0.010271
30067,1037.5300,0.074609
13135,1097.2795,0.025742
30382,1776.1620,0.385510
30016,240.8600,0.206551
13139,1083.5800,0.065171
30041,1403.3000,0.053839


In [14]:
resumen["pw"] = resumen["Precip_at_Pe85_mm"] *resumen["w"]

pa = resumen["pw"].mean()

Ac = area["A_km2"].sum() * 1000000

In [15]:
k = 0.251496377750712
ce = k * (pa - 250) / 2000 + (k - 0.15) / 1.5 
cp = pa * Ac * ce
cp

np.float64(8875141442.934872)

In [16]:
ce, Ac

(np.float64(0.05080005644514492), np.float64(1507538033.7310998))

In [17]:
Q = cp / (86400 * 365)
Q

np.float64(281.42888898195304)